In [1]:
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from itertools import combinations

## Обработка сырых данных

### Обработка резюме

In [2]:
df = pd.read_csv('data/data_rezume.csv')

In [3]:
df.head()

,id,resume_url,header_text,wrapper_text,created_at,updated_at
0,27916,https://hh.ru/resume/291373fd0002106a040039ed1...,"Был более двух недель назад\nМужчина, 42 года,...",Консультант SAP\nСпециализации:\nАналитик\nТип...,2026-04-02 04:37:13.707 +0300,2026-04-02 04:37:13.707 +0300
1,27917,https://hh.ru/resume/b353f4b100013b6a490039ed1...,"Был более двух недель назад\nМужчина, 35 лет, ...",Разработчик\nСпециализации:\nРуководитель прое...,2026-04-02 04:37:23.811 +0300,2026-04-02 04:37:23.811 +0300
2,27918,https://hh.ru/resume/45900bdc0000b094d40039ed1...,"Был более двух недель назад\nМужчина, 63 года,...",Менеджер проекта\n2 000 $ на руки\nСпециализац...,2026-04-02 04:37:31.805 +0300,2026-04-02 04:37:31.805 +0300
3,27919,https://hh.ru/resume/bd626a010007f996ce0039ed1...,"Был более двух недель назад\nМужчина, 39 лет, ...",Project/product manager\n200 000 ₽ на руки\nСп...,2026-04-02 04:37:39.382 +0300,2026-04-02 04:37:39.382 +0300
4,27920,https://hh.ru/resume/a304a31d00038f410d0039ed1...,"Была более двух недель назад\nЖенщина, 42 года...",Консультант SAP BO\nСпециализации:\nАналитик\n...,2026-04-02 04:37:47.160 +0300,2026-04-02 04:37:47.160 +0300


In [4]:
df.shape

(97849, 6)

Из текста карточек резюме извлекаем следующие атрибуты:
- Название вакансии resume_title
- Навыки, указанные в резюме skills_list
- Опыт работы experience_text
- Специализации в резюме specializations_list
- Гражданство citizenship

In [5]:
df['resume_title'] = (
    df['wrapper_text']
    .fillna('')
    .astype(str)
    .str.split('\n')
    .str[0]
    .str.strip()
)

In [6]:
def extract_skills(text):
    text = str(text)
    match = re.search(r'Навыки\s*(.*?)\s*Обо мне', text, flags=re.S)
    if not match:
        return None
    block = match.group(1).strip()
    skills = []
    for line in block.split('\n'):
        line = line.strip()
        if not line:
            continue
        if line == 'Уровни владения навыками':
            continue
        skills.append(line)
    return skills if skills else None

df['skills_list'] = df['wrapper_text'].apply(extract_skills)

In [7]:
def extract_skills(text):
    if pd.isna(text):
        return None
    
    text = str(text)

    start_match = re.search(r'(?m)^Навыки\s*$', text)
    if not start_match:
        return None

    start = start_match.end()

    end_match = re.search(
        r'(?m)^(Обо мне|Опыт вождения|Высшее образование.*|Знание языков|Гражданство, время в пути до работы)\s*$',
        text[start:]
    )

    if end_match:
        block = text[start:start + end_match.start()]
    else:
        block = text[start:]

    lines = []
    skip_lines = {
        'Уровни владения навыками',
        'Продвинутый уровень',
        'Средний уровень',
        'Начальный уровень'
    }

    for line in block.split('\n'):
        line = line.strip()
        if not line:
            continue
        if line in skip_lines:
            continue
        lines.append(line)

    skills = list(dict.fromkeys(lines))

    return skills if skills else None

df['skills_list'] = df['wrapper_text'].apply(extract_skills)

In [8]:
def extract_specializations(text):
    text = str(text)
    match = re.search(
        r'Специализац(?:ия|ии):\s*(.*?)\nТип занятости',
        text,
        flags=re.S
    )
    if not match:
        return None
    block = match.group(1).strip()   
    specializations = []
    for line in block.split('\n'):
        line = line.strip()
        if line:
            specializations.append(line)
    return specializations if specializations else None

df['specializations_list'] = df['wrapper_text'].apply(extract_specializations)

In [9]:
def extract_experience_text(text):
    text = str(text)
    match = re.search(
        r'Опыт работы\s+(\d+)\s+(?:год|года|лет)\s+(\d+)\s+(?:месяц|месяца|месяцев)',
        text
    )
    if match:
        years = int(match.group(1))
        months = int(match.group(2))
        return round(years + months / 12, 2)
    return None

df['experience_text'] = df['wrapper_text'].apply(extract_experience_text)

In [10]:
def extract_citizenship(text):
    text = str(text)
    match = re.search(
        r'Гражданство, время в пути до работы\s*\nГражданство:\s*(.*)',
        text
    )
    if match:
        return match.group(1).strip()
    return None

df['citizenship'] = df['wrapper_text'].apply(extract_citizenship)

#### Отбор релевантных резюме

Так как в датасете встречается очень много нерелевантных резюме (не относящихся к области анализа данных и работы с данными), необходимо оставить только нужные

Для начала оставим только те резюме, в названии которых присутствуют заголовки, используемые в парсинге (без учета регистра, если хотя бы заголовок парсинга просто встречался в заголовке вакансии)

In [11]:
prof_list = [
    'Аналитик DWH',
    'SQL Analyst',
    'Аналитик больших данных',
    'Специалист по анализу данных',
    'Data Scientist',
    'Data Engineer',
    'BI Developer',
    'ML Engineer',
    'Аналитик данных',
    'Data Analyst',
    'BI-аналитик',
    'Продуктовый аналитик'
]

pattern = r'(^|\b)(' + '|'.join(re.escape(x) for x in prof_list) + r')($|\b)'

mask = df['resume_title'].fillna('').str.contains(
    pattern,
    case=False,
    regex=True
)

df_new = df[mask].copy()

/var/folders/h7/k0zwpy4s6tz5mbdzhg164w_m0000gn/T/ipykernel_88702/2903095158.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df['resume_title'].fillna('').str.contains(


In [12]:
df_rez = df_new.copy()

### Обработка вакансий

In [13]:
df = pd.read_csv('data/vacant.csv')

In [14]:
df.head()

,id,vacancy_url,header_text,wrapper_text,created_at,updated_at
0,1,https://hh.ru/vacancy/129575262,Старший разработчик в группу GPU-инфраструктур...,Yandex\nInfrastructure\nМы создаём и развиваем...,2026-03-17 21:54:16.843,2026-03-17 21:54:16.843
1,2,https://hh.ru/vacancy/129454986,Project manager по развитию и трансформации би...,Обязанности:\nВыполнение функций project manag...,2026-03-17 21:54:39.568,2026-03-17 21:54:39.568
2,3,https://hh.ru/vacancy/111241834,QA Engineer (со знанием немецкого языка)\nУров...,Team.Inno – одна из наиболее опытных белорусск...,2026-03-17 21:55:04.099,2026-03-17 21:55:04.099
3,4,https://hh.ru/vacancy/129422503,Менеджер по маркетплейсу OZON\nВ архиве с 11 м...,Привет!\nМеня зовут Максим - я селлер на ВБ и ...,2026-03-17 21:55:33.566,2026-03-17 21:55:33.566
4,5,https://hh.ru/vacancy/129356560,Менеджер по закупкам и снабжению\nВ архиве с 1...,Tonka Perfumes Moscow – российский парфюмерный...,2026-03-17 21:56:04.140,2026-03-17 21:56:04.140


In [15]:
df["header_text_list"] = df["header_text"].fillna("").str.split("\n")

In [16]:
df["header_text_list"] = df["header_text_list"].apply(
    lambda x: [item for item in x if not (isinstance(item, str) and item.startswith("В архиве"))]
    if isinstance(x, list) else x
)

Из текста карточек вакансий извлекаем следующие атрибуты:
 - Наименование vacancy_name
 - Опыт работы experience
 - Указанная вилка по з/п salary
 - Флаг присутствия вилки по з/п salary_exists, от и до salary_from, salary_to
 - Форматт работы work_format
 - Тип занятости employment_type
 - Требуемые навыки skills_list
 - Опыт работы (от скольки лет, числовое представвление) experience_years_min

In [17]:
df["vacancy_name"] = df["header_text_list"].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
)

In [18]:
df["experience"] = df["header_text_list"].apply(
    lambda x: next(
        (
            item.replace("Опыт работы:", "").strip()
            for item in x
            if isinstance(item, str) and item.startswith("Опыт работы:")
        ),
        None
    ) if isinstance(x, list) else None
)

In [19]:
df["salary"] = df["header_text_list"].apply(
    lambda x: x[1] if isinstance(x, list) and len(x) > 1 else None
)

In [20]:
def parse_salary(text):
    if pd.isna(text) or str(text).strip() == "Уровень дохода не указан":
        return pd.Series([False, None, None])

    text = str(text).strip()

    numbers = re.findall(r"\d[\d\s]*", text)
    numbers = [int(num.replace(" ", "")) for num in numbers]

    if len(numbers) == 0:
        return pd.Series([False, None, None])

    if len(numbers) == 1:
        number = numbers[0]

        if text.startswith("от"):
            return pd.Series([True, number, None])

        if text.startswith("до"):
            return pd.Series([True, None, number])

        return pd.Series([True, number, number])

    return pd.Series([True, numbers[0], numbers[1]])

df[["salary_exists", "salary_from", "salary_to"]] = df["salary"].apply(parse_salary)

In [21]:
df["work_format"] = df["header_text_list"].apply(
    lambda x: x[-1].replace("Формат работы:", "").strip()
    if isinstance(x, list) and len(x) > 0 and isinstance(x[-1], str) and x[-1].startswith("Формат работы:")
    else None
)

In [22]:
df["employment_type"] = df["header_text_list"].apply(
    lambda x: next(
        (item for item in x if isinstance(item, str) and "занятость" in item.lower()),
        None
    ) if isinstance(x, list) else None
)

In [23]:
def parse_skills(text):
    if pd.isna(text):
        return []
    
    text = str(text)
    
    match = re.search(
        r"Ключевые навыки\s*(.*?)\s*Где предстоит работать",
        text,
        flags=re.S
    )
    
    if not match:
        return []
    
    skills_block = match.group(1).strip()
    
    skills = [
        line.strip()
        for line in skills_block.split("\n")
        if line.strip()
    ]
    
    return skills

df["skills_list"] = df["wrapper_text"].apply(parse_skills)

In [24]:
def parse_experience(text):
    if pd.isna(text):
        return None

    text = str(text).lower().strip().replace('–', '-')

    if 'нет опыта' in text:
        return 0
    if 'более 6' in text:
        return 6

    match = re.search(r'(\d+)\s*-\s*(\d+)', text)
    if match:
        return int(match.group(1))

    match = re.search(r'от\s*(\d+)', text)
    if match:
        return int(match.group(1))

    return None

df['experience_years_min'] = df['experience'].apply(parse_experience)

#### Отбор релевантных вакансий

Так же, как и с резюме, при парсинге получили большое количество нерелевантных вакансий

Поэтому поступим для начала так же - отберем вакансии по заголовку парсинга

In [25]:
prof_list = [
    'Аналитик DWH',
    'SQL Analyst',
    'Аналитик больших данных',
    'Специалист по анализу данных',
    'Data Scientist',
    'Data Engineer',
    'BI Developer',
    'ML Engineer',
    'Аналитик данных',
    'Data Analyst',
    'BI-аналитик',
    'Продуктовый аналитик'
]

pattern = r'(^|\b)(' + '|'.join(re.escape(x) for x in prof_list) + r')($|\b)'

mask = df['vacancy_name'].fillna('').str.contains(
    pattern,
    case=False,
    regex=True
)

df_new = df[mask].copy()

/var/folders/h7/k0zwpy4s6tz5mbdzhg164w_m0000gn/T/ipykernel_88702/3057247236.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df['vacancy_name'].fillna('').str.contains(


In [26]:
df_vac = df_new.copy()

In [27]:
df_vac = df_vac[
    df_vac["skills_list"].apply(lambda x: isinstance(x, list) and len(x) != 0)
]

## EDA по полученным датасетам

In [28]:
df_rez.head()

,id,resume_url,header_text,wrapper_text,created_at,updated_at,resume_title,skills_list,specializations_list,experience_text,citizenship
14,27934,https://hh.ru/resume/cbbdd4310006da6fba0039ed1...,"Был более двух недель назад\nМужчина, 26 лет, ...",Data Scientist\nСпециализации:\nАналитик\nДата...,2026-04-02 04:40:04.940 +0300,2026-04-02 04:40:04.940 +0300,Data Scientist,"[Python, Анализ данных, SQL, Теория вероятност...","[Аналитик, Дата-сайентист]",NaN,Россия
36,27956,https://hh.ru/resume/0e9e351a0001d2d0b80039ed1...,"Was more than two weeks ago\nMale, 40 years, b...","Power BI Certified, Tableau, Manager BI & Anal...",2026-04-02 04:43:33.054 +0300,2026-04-02 04:43:33.054 +0300,"Power BI Certified, Tableau, Manager BI & Anal...",None,None,NaN,None
39,27959,https://hh.ru/resume/5dc6fedb00041e697c0039ed1...,"Был более двух недель назад\nМужчина, 33 года,...",Lead Data Scientist\n6 000 € на руки\nСпециали...,2026-04-02 04:43:58.816 +0300,2026-04-02 04:43:58.816 +0300,Lead Data Scientist,"[machine learning, Python, neural networks, Сб...","[Аналитик, Дата-сайентист, Директор по информа...",14.08,Россия
40,27960,https://hh.ru/resume/bb05cbaf0000ccdc6f0039ed1...,"Был более двух недель назад\nМужчина, 56 лет, ...","Data engineer\nСпециализации:\nПрограммист, ра...",2026-04-02 04:44:05.223 +0300,2026-04-02 04:44:05.223 +0300,Data engineer,"[C#, ETL, Reporting, Transact-SQL, Denodo, Pyt...","[Программист, разработчик, Руководитель группы...",23.33,Россия
43,27963,https://hh.ru/resume/0f609f880003876c9f0039ed1...,"Был более двух недель назад\nМужчина, 45 лет, ...",Data engineer\n200 000 ₽ на руки\nСпециализаци...,2026-04-02 04:44:29.672 +0300,2026-04-02 04:44:29.672 +0300,Data engineer,"[SQL, Уровень не указан, HTML, JavaScript, XML...","[Дата-сайентист, Программист, разработчик, Рук...",23.33,Россия


In [29]:
df_vac.head()

,id,vacancy_url,header_text,wrapper_text,created_at,updated_at,header_text_list,vacancy_name,experience,salary,salary_exists,salary_from,salary_to,work_format,employment_type,skills_list,experience_years_min
34,34,https://hh.ru/vacancy/129853395,Data Engineer (разработчик DWH)\nВ архиве с 17...,Задаем тренды в технологиях ритейла\nX5 Group ...,2026-03-17 22:13:18.663,2026-03-17 22:13:18.663,"[Data Engineer (разработчик DWH), Уровень дохо...",Data Engineer (разработчик DWH),3–6 лет,Уровень дохода не указан,False,NaN,NaN,"на месте работодателя, удалённо или гибрид",Полная занятость,"[SQL, Python, Big Data, Apache Airflow, Apache...",3.0
64,162,https://hh.ru/vacancy/130038332,Data Engineer по построению DWH\nУровень доход...,Компания EcoFinance развивает и внедряет проду...,2026-03-17 23:43:51.117,2026-03-17 23:43:51.117,"[Data Engineer по построению DWH, Уровень дохо...",Data Engineer по построению DWH,3–6 лет,Уровень дохода не указан,False,NaN,NaN,гибрид,Полная занятость,"[ETL, SQL, DWH, Apache Kafka, Debezium, BI, db...",3.0
114,113,https://hh.ru/vacancy/130129118,Middle Data Analyst\nВ архиве с 5 марта 2026\n...,Мы Kaspi.kz - крупнейшая технологическая компа...,2026-03-17 23:08:14.387,2026-03-17 23:08:14.387,"[Middle Data Analyst, Уровень дохода не указан...",Middle Data Analyst,1–3 года,Уровень дохода не указан,False,NaN,NaN,на месте работодателя,Полная занятость,"[Python, SQL, Анализ данных, Big Data, Apache ...",1.0
195,192,https://hh.ru/vacancy/129577973,Senior Big Data Engineer\nВ архиве с 14 феврал...,A publicly traded technology consulting and en...,2026-03-18 00:00:55.009,2026-03-18 00:00:55.009,"[Senior Big Data Engineer, Уровень дохода не у...",Senior Big Data Engineer,3–6 лет,Уровень дохода не указан,False,NaN,NaN,гибрид,Полная занятость,"[Scala, Python, Apache Spark, Jupyter Notebook...",3.0
233,266,https://hh.ru/vacancy/130316419,Аналитик данных Центра развития электронных об...,Обязанности:\n1. Проводить аналитическую работ...,2026-03-18 00:44:19.458,2026-03-18 00:44:19.458,[Аналитик данных Центра развития электронных о...,Аналитик данных Центра развития электронных об...,1–3 года,от 50 000 ₽ за месяц до вычета налогов,True,50000.0,NaN,на месте работодателя,Полная занятость,"[Анализ данных, Базы данных, Деловое общение, ...",1.0


In [30]:
(df_vac["skills_list"].apply(lambda x: isinstance(x, list) and len(x) != 0)).sum()

np.int64(571)

### Удаление технических столбцов

Для baseline оставим только столбцы, необходимые для сопоставления

In [31]:
df_rez_base = df_rez[['id', 'resume_title', 'skills_list', 'experience_text', 'wrapper_text']]
df_vac_base = df_vac[['id', 'vacancy_name', 'skills_list', 'experience_years_min', 'wrapper_text']]

### Базовые характеристики

In [33]:
display(df_rez_base.dtypes.reset_index().rename(columns={'index': 'column', 0: 'dtype'}))
display(df_vac_base.dtypes.reset_index().rename(columns={'index': 'column', 0: 'dtype'}))

,column,dtype
0,id,int64
1,resume_title,object
2,skills_list,object
3,experience_text,float64
4,wrapper_text,object


,column,dtype
0,id,int64
1,vacancy_name,object
2,skills_list,object
3,experience_years_min,float64
4,wrapper_text,object


### Проверка и обработка пропусков

In [34]:
df_rez_base.isna().sum().sort_values(ascending=False)

experience_text    2549
skills_list        1289
id                    0
resume_title          0
wrapper_text          0
dtype: int64

In [35]:
df_vac_base.isna().sum().sort_values(ascending=False)

experience_years_min    34
id                       0
vacancy_name             0
skills_list              0
wrapper_text             0
dtype: int64

In [36]:
df_rez_base[df_rez_base.isna().any(axis=1)]

,id,resume_title,skills_list,experience_text,wrapper_text
14,27934,Data Scientist,"[Python, Анализ данных, SQL, Теория вероятност...",NaN,Data Scientist\nСпециализации:\nАналитик\nДата...
36,27956,"Power BI Certified, Tableau, Manager BI & Anal...",None,NaN,"Power BI Certified, Tableau, Manager BI & Anal..."
85,28005,Data Engineer,"[MS Outlook, Пользователь ПК, Internet, Англий...",NaN,"Data Engineer\nСпециализации:\nBI-аналитик, ан..."
90,32919,Data Scientist,None,NaN,Data Scientist\nSpecializations:\nAnalyst\nDat...
223,28142,Data Analyst,None,NaN,"Data Analyst\nSpecializations:\nBI analyst, da..."
...,...,...,...,...,...
97763,27837,"Data Architect, ML Engineer, Data Engineer, Da...",None,NaN,"Data Architect, ML Engineer, Data Engineer, Da..."
97774,27847,"Data Engineer, Data Science",None,NaN,"Data Engineer, Data Science\nSpecializations:\..."
97805,27877,Data Engineer,None,NaN,Data Engineer\n1 000 000 ₸ in hand\nSpecializa...
97831,32905,MLOps / ML Engineer (EN),None,NaN,MLOps / ML Engineer (EN)\nSpecializations:\nDa...


Видим пропуски в опыте работы

Логично считать, что кандидаты не заполняют опыт работы при его отсутствии, поэтому заполним нулями

In [37]:
df_rez_base['experience_text'] = df_rez_base['experience_text'].fillna(0)

/var/folders/h7/k0zwpy4s6tz5mbdzhg164w_m0000gn/T/ipykernel_88702/1807016381.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_rez_base['experience_text'] = df_rez_base['experience_text'].fillna(0)


In [38]:
df_vac_base[df_vac_base.isna().any(axis=1)]

,id,vacancy_name,skills_list,experience_years_min,wrapper_text
576,568,Аналитик данных по работе с маркетплейсами (ст...,"[Ехсеl, Microsoft Power BI, сводные таблицы, ф...",NaN,CADesign — ведущая компания с 15-летним опытом...
584,576,Junior BI-аналитик,"[SQL, Business Intelligence Systems, Power BI,...",NaN,Junior BI-аналитик (частичная занятость)\nАльф...
952,944,Junior продуктовый аналитик,"[Аналитический склад ума, Исследовательский ан...",NaN,"Привет, будущий коллега!\nМы — IT компания Фор..."
2083,2072,"Стажер - Data Scientist, Ozon Банк",[Data Science],NaN,стартап вырос в банк с\nмлн клиентов\nМы строи...
2266,2252,Аналитик данных,"[Анализ данных, Маркетинговые метрики, XML, Ор...",NaN,Обязанности:\n• Анализ и обработка данных\n• У...
2830,2819,Начинающий аналитик данных,"[MS Excel, VBA, Работа с большим объемом инфор...",NaN,"Вы начинающий специалист, мечтающий построить ..."
3213,3180,BI-аналитик,"[Power BI, Статистический анализ, Бизнес-анали...",NaN,С НАМИ ЭФФЕКТИВНЕЕ!\nИстория компании AKFA — э...
3665,3629,Продуктовый аналитик / Маркетолог аналитик,"[Аналитические исследования, Работа с большим ...",NaN,Tekna Line – торгово-производственная компания...
4008,3971,Специалист по работе с таблицами/аналитик данн...,"[MS Excel, Внимательность, Стрессоустойчивость...",NaN,Обязанности:\nСравнение актуальных цен и марки...
4239,4205,Junior Data Analyst (Автоматизация складов),"[SQL, Jupyter Notebook, A/B тесты]",NaN,Объединённая компания Wildberries и Russ — это...


Видим 1 строку по вакансиям с неуказанным опытом работы

Пока заполним его нулем

In [39]:
df_vac_base['experience_years_min'] = df_vac_base['experience_years_min'].fillna(0)

/var/folders/h7/k0zwpy4s6tz5mbdzhg164w_m0000gn/T/ipykernel_88702/1567853829.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_vac_base['experience_years_min'] = df_vac_base['experience_years_min'].fillna(0)


## Построение baseline

In [40]:
df_rez_base = df_rez_base[['id', 'resume_title', 'skills_list', 'experience_text']].copy()
df_vac_base = df_vac_base[['id', 'vacancy_name', 'skills_list', 'experience_years_min']].copy()

In [94]:
df_rez_base.shape

(11782, 4)

In [95]:
df_vac_base.shape

(571, 4)

### Формирование полуэталона - построение range на основе навыков совпадающих навыков и опыта работы

Для данной задачи был использован следующий метод:

Для расчета score по опыту работы
$$
exp\_score =
\begin{cases}
1, & \text{если } resume\_exp \ge vacancy\_exp \\
\dfrac{resume\_exp}{vacancy\_exp}, & \text{если } resume\_exp < vacancy\_exp
\end{cases}
$$

При расчете score по навыкам:
$$
skill\_score =
\frac{|resume\_skills \cap vacancy\_skills|}{|vacancy\_skills|}
$$

Итоговый score

$$
final\_score = 0.67 \cdot skill\_score + 0.33 \cdot exp\_score
$$


In [41]:
def skill_match_score(resume_skills, vacancy_skills):
    resume_set = set(resume_skills) if isinstance(resume_skills, list) else set()
    vacancy_set = set(vacancy_skills) if isinstance(vacancy_skills, list) else set()

    if len(vacancy_set) == 0:
        return 0.0, 0

    matched_count = len(resume_set & vacancy_set)
    skill_score = matched_count / len(vacancy_set)

    return skill_score, matched_count


def exp_match_score(resume_exp, vacancy_exp):
    if pd.isna(resume_exp):
        resume_exp = 0
    if pd.isna(vacancy_exp):
        vacancy_exp = 0

    if vacancy_exp == 0:
        return 1.0

    if resume_exp >= vacancy_exp:
        return 1.0

    return resume_exp / vacancy_exp


def final_match_score(resume_skills, vacancy_skills, resume_exp, vacancy_exp):
    skill_score, matched_count = skill_match_score(resume_skills, vacancy_skills)
    exp_score = exp_match_score(resume_exp, vacancy_exp)

    final_score = (2/3) * skill_score + (1/3) * exp_score

    return {
        'skill_score': skill_score,
        'matched_skills_count': matched_count,
        'exp_score': exp_score,
        'final_score': final_score
    }

In [42]:
def rank_vacancies_for_resume(
    resume_row,
    df_vac,
    resume_skills_col='skills_list',
    resume_exp_col='experience_text',
    vacancy_skills_col='skills_list',
    vacancy_exp_col='experience_years_min'
):
    results = []

    for _, vac_row in df_vac.iterrows():
        scores = final_match_score(
            resume_skills=resume_row[resume_skills_col],
            vacancy_skills=vac_row[vacancy_skills_col],
            resume_exp=resume_row[resume_exp_col],
            vacancy_exp=vac_row[vacancy_exp_col]
        )

        results.append({
            'resume_id': resume_row['id'],
            'resume_title': resume_row['resume_title'],
            'vacancy_id': vac_row['id'],
            'vacancy_name': vac_row['vacancy_name'],
            'vacancy_exp': vac_row[vacancy_exp_col],
            'skill_score': scores['skill_score'],
            'matched_skills_count': scores['matched_skills_count'],
            'exp_score': scores['exp_score'],
            'final_score': scores['final_score']
        })

    result_df = pd.DataFrame(results)

    result_df = result_df.sort_values(
        by=['final_score', 'matched_skills_count'],
        ascending=[False, False]
    ).reset_index(drop=True)

    result_df['rank'] = range(1, len(result_df) + 1)

    return result_df

In [43]:
all_matches = []

for _, resume_row in df_rez_base.iterrows():
    ranked = rank_vacancies_for_resume(
        resume_row=resume_row,
        df_vac=df_vac_base,
        resume_skills_col='skills_list',
        resume_exp_col='experience_text',
        vacancy_skills_col='skills_list',
        vacancy_exp_col='experience_years_min'
    )
    all_matches.append(ranked)

matches_df = pd.concat(all_matches, ignore_index=True)
matches_df.head()

,resume_id,resume_title,vacancy_id,vacancy_name,vacancy_exp,skill_score,matched_skills_count,exp_score,final_score,rank
0,27934,Data Scientist,4486,Junior Data Scientist,0.0,1.000000,1,1.0,1.000000,1
1,27934,Data Scientist,10404,Аналитик данных / Специалист по CRM,0.0,1.000000,1,1.0,1.000000,2
2,27934,Data Scientist,6769,Junior Data Engineer / Разработчик ETL,0.0,0.666667,2,1.0,0.777778,3
3,27934,Data Scientist,4676,Data Scientist,3.0,1.000000,3,0.0,0.666667,4
4,27934,Data Scientist,6183,Аналитик данных,1.0,1.000000,3,0.0,0.666667,5


In [44]:
matches_df.shape

(6727522, 10)

In [45]:
def select_top_matches(group, top_unique_scores=5):
    group = group.sort_values(
        by=["final_score", "matched_skills_count", "vacancy_exp"],
        ascending=[False, False, False]
    ).copy()

    top_scores = (
        group.loc[group["final_score"] > 0, "final_score"]
        .drop_duplicates()
        .head(top_unique_scores)
    )

    filtered = group[group["final_score"].isin(top_scores)].copy()

    filtered = filtered.sort_values(
        by=["final_score", "matched_skills_count", "vacancy_exp"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    filtered["rank"] = range(1, len(filtered) + 1)

    return filtered

In [46]:
result = (
    matches_df
    .groupby("resume_id", group_keys=False)
    .apply(select_top_matches)
    .reset_index(drop=True)
)

/var/folders/h7/k0zwpy4s6tz5mbdzhg164w_m0000gn/T/ipykernel_88702/559495581.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(select_top_matches)


In [47]:
cols_to_show = [
    'resume_id',
    'resume_title',
    'vacancy_id',
    'vacancy_name',
    'matched_skills_count',
    'skill_score',
    'exp_score',
    'final_score'
]

result[cols_to_show].head(30)

,resume_id,resume_title,vacancy_id,vacancy_name,matched_skills_count,skill_score,exp_score,final_score
0,2,Аналитик данных,9445,ML Engineer (LLM / RAG),1,1.000000,1.0,1.000000
1,2,Аналитик данных,4486,Junior Data Scientist,1,1.000000,1.0,1.000000
2,2,Аналитик данных,9083,Team Lead Data Analyst,4,0.800000,1.0,0.866667
3,2,Аналитик данных,4195,Data Engineer,3,0.750000,1.0,0.833333
4,2,Аналитик данных,5136,Junior Data Analyst (mobile games),3,0.750000,1.0,0.833333
5,2,Аналитик данных,7931,Аналитик данных,5,0.714286,1.0,0.809524
6,2,Аналитик данных,10143,Аналитик данных (Fraud),5,0.714286,1.0,0.809524
7,2,Аналитик данных,6507,Аналитик данных (senior),2,0.666667,1.0,0.777778
8,2,Аналитик данных,7958,Аналитик данных / Data analyst,2,0.666667,1.0,0.777778
9,2,Аналитик данных,8164,Аналитик данных Power BI,2,0.666667,1.0,0.777778


In [93]:
pseudo_counts = result.groupby("resume_id")["vacancy_id"].nunique()
pseudo_counts.describe()

count    11782.000000
mean        63.330080
std        107.564487
min          5.000000
25%         25.000000
50%         40.000000
75%         60.000000
max        571.000000
Name: vacancy_id, dtype: float64

### Построение baseline только на текстах вакансий

In [48]:
resume_texts = df_rez["wrapper_text"].tolist()
vacancy_texts = df_vac["wrapper_text"].tolist()

In [ ]:
all_texts = resume_texts + vacancy_texts

vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=2
)

all_matrix = vectorizer.fit_transform(all_texts)

resume_matrix = all_matrix[:len(resume_texts)]
vacancy_matrix = all_matrix[len(resume_texts):]

sim_matrix = cosine_similarity(resume_matrix, vacancy_matrix)

In [85]:
k = 5

top_k_indices = np.argsort(-sim_matrix, axis=1)[:, :k]
top_k_scores = np.take_along_axis(sim_matrix, top_k_indices, axis=1)

In [86]:
results = []

for i in range(len(df_rez_base)):
    for rank in range(k):
        vac_idx = top_k_indices[i, rank]
        score = top_k_scores[i, rank]

        results.append({
            "resume_id": df_rez_base.iloc[i]["id"],
            "resume_title": df_rez_base.iloc[i]["resume_title"],
            "vacancy_id": df_vac_base.iloc[vac_idx]["id"],
            "vacancy_name": df_vac_base.iloc[vac_idx]["vacancy_name"],
            "baseline_score": score,
            "rank": rank + 1
        })

baseline_result = pd.DataFrame(results)

In [87]:
baseline_result.head()

,resume_id,resume_title,vacancy_id,vacancy_name,baseline_score,rank
0,27934,Data Scientist,3155,Data scientist (ML),0.147904,1
1,27934,Data Scientist,13595,Data Analyst в управление комплаенс,0.126044,2
2,27934,Data Scientist,12776,Senior Data Scientist,0.125925,3
3,27934,Data Scientist,11068,Инженер данных (Data Engineer),0.118100,4
4,27934,Data Scientist,14846,Аналитик данных / Ad-hoc-аналитик,0.113316,5


### Проверка на метриках

#### Skills Overlap@5

Метрика показывает, насколько хорошо baseline в top-5 подбирает вакансии, у которых навыки пересекаются с навыками резюме

In [88]:
def skills_overlap_ratio(resume_skills, vacancy_skills):
    if not isinstance(resume_skills, list):
        resume_skills = []
    if not isinstance(vacancy_skills, list):
        vacancy_skills = []

    resume_set = set(str(skill).strip().lower() for skill in resume_skills if pd.notna(skill))
    vacancy_set = set(str(skill).strip().lower() for skill in vacancy_skills if pd.notna(skill))

    if len(vacancy_set) == 0:
        return 0.0

    return len(resume_set & vacancy_set) / len(vacancy_set)

resume_skills_df = df_rez_base[["id", "skills_list"]].rename(columns={
    "id": "resume_id",
    "skills_list": "resume_skills"
})

vacancy_skills_df = df_vac_base[["id", "skills_list"]].rename(columns={
    "id": "vacancy_id",
    "skills_list": "vacancy_skills"
})

baseline_eval = (
    baseline_result
    .merge(resume_skills_df, on="resume_id", how="left")
    .merge(vacancy_skills_df, on="vacancy_id", how="left")
)

baseline_eval["skills_overlap"] = baseline_eval.apply(
    lambda row: skills_overlap_ratio(row["resume_skills"], row["vacancy_skills"]),
    axis=1
)

skills_overlap_5 = (
    baseline_eval
    .sort_values(["resume_id", "baseline_score"], ascending=[True, False])
    .groupby("resume_id")
    .head(5)
    .groupby("resume_id")["skills_overlap"]
    .mean()
    .reset_index(name="skills_overlap@5")
)

skills_overlap_summary = (
    skills_overlap_5
    .drop(columns="resume_id")
    .agg(["mean", "median", "std", "min", "max"])
    .T
)

display(skills_overlap_5.head())
display(skills_overlap_summary)

,resume_id,skills_overlap@5
0,2,0.172683
1,3,0.164698
2,4,0.056190
3,6,0.548578
4,8,0.373810


,mean,median,std,min,max
skills_overlap@5,0.230446,0.218889,0.178276,0.0,0.905556


#### Intersection Count@5

Метрика показывает, сколько вакансий из top-5 baseline пересеклись с полуэталоном

In [89]:
pseudo_top = result.copy()

baseline_top5 = (
    baseline_result
    .sort_values(["resume_id", "rank"], ascending=[True, True])
    .groupby("resume_id")
    .head(5)
)

pseudo_lists = pseudo_top.groupby("resume_id")["vacancy_id"].apply(set)
baseline_lists = baseline_top5.groupby("resume_id")["vacancy_id"].apply(set)

comparison = pd.concat([pseudo_lists, baseline_lists], axis=1).dropna()
comparison.columns = ["pseudo_vacancies", "baseline_vacancies"]

comparison["intersection_count@5"] = comparison.apply(
    lambda row: len(row["pseudo_vacancies"] & row["baseline_vacancies"]),
    axis=1
)

intersection_summary = (
    comparison["intersection_count@5"]
    .agg(["mean", "median", "std", "min", "max"])
    .to_frame(name="intersection_count@5")
)

display(comparison.head())
display(intersection_summary)

,pseudo_vacancies,baseline_vacancies,intersection_count@5
resume_id,,,
2,"{4195, 8164, 9445, 4486, 6507, 4205, 5136, 110...","{7175, 14708, 6645, 6454, 15447}",0
3,"{12163, 2693, 7174, 2311, 4486, 4363, 654, 513...","{13415, 10505, 10348, 11990, 15447}",0
4,"{736, 7042, 10404, 7174, 6183, 14664, 4363, 16...","{13671, 3629, 10100, 10165, 7959}",0
6,"{576, 2819, 10404, 12901, 4486, 12676, 9307, 1...","{8164, 14708, 6645, 6454, 14846}",0
8,"{12163, 7428, 11523, 4486, 2311, 10129, 14099,...","{6470, 266, 5999, 8475, 13595}",1


,intersection_count@5
mean,0.633933
median,0.000000
std,1.131614
min,0.000000
max,5.000000


#### Mean Top-5 Score

Средний baseline_score по top-5.

In [90]:
top5 = (
    baseline_result
    .sort_values(["resume_id", "baseline_score"], ascending=[True, False])
    .groupby("resume_id")
    .head(5)
)

mean_top5_score = (
    top5
    .groupby("resume_id")["baseline_score"]
    .mean()
    .reset_index(name="mean_top5_score")
)

mean_top5_summary = (
    mean_top5_score["mean_top5_score"]
    .agg(["mean", "median", "std", "min", "max"])
    .to_frame(name="mean_top5_score")
)

display(mean_top5_score.head())
display(mean_top5_summary)

,resume_id,mean_top5_score
0,2,0.136121
1,3,0.146077
2,4,0.109663
3,6,0.260188
4,8,0.141111


,mean_top5_score
mean,0.163351
median,0.136942
std,0.094343
min,0.033128
max,0.633860


#### Intra-list Diversity@5

Показывает, насколько вакансии внутри одного top-5 похожи друг на друга

In [91]:
def jaccard_similarity(skills_a, skills_b):
    if not isinstance(skills_a, list):
        skills_a = []
    if not isinstance(skills_b, list):
        skills_b = []

    set_a = set(str(skill).strip().lower() for skill in skills_a if pd.notna(skill))
    set_b = set(str(skill).strip().lower() for skill in skills_b if pd.notna(skill))

    union = set_a | set_b
    intersection = set_a & set_b

    if len(union) == 0:
        return 0.0

    return len(intersection) / len(union)

vacancy_skills_df = df_vac_base[["id", "skills_list"]].rename(columns={
    "id": "vacancy_id",
    "skills_list": "vacancy_skills"
})

baseline_eval = baseline_result.merge(
    vacancy_skills_df,
    on="vacancy_id",
    how="left"
)

top5 = (
    baseline_eval
    .sort_values(["resume_id", "baseline_score"], ascending=[True, False])
    .groupby("resume_id")
    .head(5)
)

rows = []

for resume_id, group in top5.groupby("resume_id"):
    vacancy_skills_lists = group["vacancy_skills"].tolist()

    pair_diversities = []

    for skills_a, skills_b in combinations(vacancy_skills_lists, 2):
        sim = jaccard_similarity(skills_a, skills_b)
        div = 1 - sim
        pair_diversities.append(div)

    if len(pair_diversities) == 0:
        diversity = 0.0
    else:
        diversity = sum(pair_diversities) / len(pair_diversities)

    rows.append({
        "resume_id": resume_id,
        "intra_list_diversity@5": diversity
    })

intra_list_diversity_5 = pd.DataFrame(rows)

intra_list_diversity_summary = (
    intra_list_diversity_5["intra_list_diversity@5"]
    .agg(["mean", "median", "std", "min", "max"])
    .to_frame(name="intra_list_diversity@5")
)

display(intra_list_diversity_5.head())
display(intra_list_diversity_summary)

,resume_id,intra_list_diversity@5
0,2,0.843273
1,3,0.861111
2,4,0.945846
3,6,0.854728
4,8,0.916111


,intra_list_diversity@5
mean,0.873128
median,0.892933
std,0.090014
min,0.000000
max,0.997674


## Вывод

- Были обработаны сырые тексты парсинга карточек резюме и вакансий с HeadHunter
- В результате были получены 11782 резюме и 571 вакансия (при первичной обработке, при фильтрации по указанным навыкам)
- был построен "псевдоэталон" - расширенное множество потенциально релевантных вакансий, основанных на пересечении навыков (в большей степени) и соответствии опыта (в меньшей)
- На подходе TF-IDF + cosine similarity по полным текстам был построен baseline 
- Оценка качества показала, что данный baseline находит только близкие вакансии с ограниченным качеством - стоит использовать более продвинутые методы сопоставления
